In [2]:
### SQL Database Parsing
#Create a SQLite database and insert the data from the DataFrame into a table named 'products'. Then, query the database to retrieve all records from the 'products' table and display them.
import sqlite3
import os

os.makedirs('data/databases', exist_ok=True)

In [3]:
# create sample Database
conn = sqlite3.connect('data/databases/company_data.db')
cursor = conn.cursor()


In [4]:
# Create a table named 'employees' with appropriate columns
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        position TEXT,
        department TEXT,
        salary REAL
    )
''')

In [5]:
# Create a table named 'project' with appropriate columns
cursor.execute('''
    CREATE TABLE IF NOT EXISTS project (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        lead_id INTEGER,
        start_date TEXT,
        end_date TEXT,
        budget REAL
    )
''')

In [6]:
# insert sample data into the 'employees' table
employees_data = [
    (1, "John Doe", "Software Engineer", "Engineering", 75000.0),
    (2, "Jane Smith", "Product Manager", "Product", 85000.0),
    (3, "Raja Govindan", "Sales Associate", "Sales", 55000.0)
]
cursor.executemany('''
    INSERT OR REPLACE INTO employees (id, name, position, department, salary)
    VALUES (?, ?, ?, ?, ?)
''', employees_data)

# insert sample data into the 'project' table
project_data = [
    (1, "Project Alpha", 1, "2023-01-01", "2023-12-31", 100000.0),
    (2, "Project Beta", 2, "2023-02-01", "2023-11-30", 150000.0),
    (3, "Project Gamma", 3, "2023-03-01", "2023-10-31", 200000.0)
]
cursor.executemany('''
    INSERT OR REPLACE INTO project (id, name, lead_id, start_date, end_date, budget)
    VALUES (?, ?, ?, ?, ?, ?)
''', project_data)

# Commit the changes and close the connection
conn.commit()
conn.close()

In [12]:
conn = sqlite3.connect('data/databases/company_data.db')
cursor = conn.cursor()
print(cursor.execute('SELECT * FROM employees').fetchall()[0])

(1, 'John Doe', 'Software Engineer', 'Engineering', 75000.0)


In [14]:
### Database content extraction
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

c:\Users\rajam\learnings\project\RAGDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# Method 1: Using SQLDatabase Utility
db = SQLDatabase.from_uri("sqlite:///data/databases/company_data.db")
print(db.get_usable_table_names())
print(db.get_table_info())


['employees', 'project']

CREATE TABLE employees (
	id INTEGER, 
	name TEXT NOT NULL, 
	position TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	position	department	salary
1	John Doe	Software Engineer	Engineering	75000.0
2	Jane Smith	Product Manager	Product	85000.0
3	Raja Govindan	Sales Associate	Sales	55000.0
*/


CREATE TABLE project (
	id INTEGER, 
	name TEXT NOT NULL, 
	lead_id INTEGER, 
	start_date TEXT, 
	end_date TEXT, 
	budget REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from project table:
id	name	lead_id	start_date	end_date	budget
1	Project Alpha	1	2023-01-01	2023-12-31	100000.0
2	Project Beta	2	2023-02-01	2023-11-30	150000.0
3	Project Gamma	3	2023-03-01	2023-10-31	200000.0
*/


In [40]:
# Custom sql to document conversion
from typing import List
from langchain_core.documents import Document
print("Custom sql to document conversion")

def sql_to_documents(db_path: str, db) -> List[Document]:
    """Convert sql database into documents with context"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []

    # Strategy:1, create documents for each table with context
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()
        columns = [description[0] for description in cursor.description]

        # Create table overview document
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(columns)}\n"
        table_content += f"Total Records: {len(rows)}\n"

        # Add sample records to the document
        table_content += "\nSample Records:\n"
        for row in rows:
            table_content += f"Values: {', '.join(str(val) for val in row)}\n"

        doc = Document(
            page_content=table_content,
            metadata={
                "table_name": f"table_{table_name}", 
                "source": f"sqlite:///{db_path}",
                "data_type": "database",
                "num_records": len(rows)
            }
        )
        documents.append(doc)

    # Strategy:2, create documents with some relationship context (e.g., foreign key relationships)
    # For simplicity, join employees and project tables based on lead_id
    relationship_query = '''
        SELECT e.name AS employee_name, e.position, p.name AS project_name, p.start_date, p.end_date
        FROM employees e
        JOIN project p ON e.id = p.lead_id;
    '''
    cursor.execute(relationship_query)
    relationship_rows = cursor.fetchall()
    relationship_content = "Employee-Project Relationships:\n"
    for row in relationship_rows:
        relationship_content += f"Employee: {row[0]}, Position: {row[1]}, Project: {row[2]}, Start Date: {row[3]}, End Date: {row[4]}\n"

    relationship_doc = Document(
        page_content=relationship_content,
        metadata={
            "table_name": "employee_project_relationships",
            "source": f"sqlite:///{db_path}",
            "data_type": "sql_relationships",
        }
    )
    documents.append(relationship_doc)
    conn.close()
    return documents

Custom sql to document conversion


In [41]:
docs = sql_to_documents('data/databases/company_data.db', db)
print(f"Total documents created: {len(docs)}")
print(docs[0].page_content)
print(docs[0].metadata)
print(docs[-1].page_content)

Total documents created: 3
Table: employees
Columns: id, name, position, department, salary
Total Records: 3

Sample Records:
Values: 1, John Doe, Software Engineer, Engineering, 75000.0
Values: 2, Jane Smith, Product Manager, Product, 85000.0
Values: 3, Raja Govindan, Sales Associate, Sales, 55000.0

{'table_name': 'table_employees', 'source': 'sqlite:///data/databases/company_data.db', 'data_type': 'database', 'num_records': 3}
Employee-Project Relationships:
Employee: John Doe, Position: Software Engineer, Project: Project Alpha, Start Date: 2023-01-01, End Date: 2023-12-31
Employee: Jane Smith, Position: Product Manager, Project: Project Beta, Start Date: 2023-02-01, End Date: 2023-11-30
Employee: Raja Govindan, Position: Sales Associate, Project: Project Gamma, Start Date: 2023-03-01, End Date: 2023-10-31

